# Empirical Asset Pricing Tests: CAPM, Fama-French 3-Factor and Carhart 4-Factor

This notebook tests how well factor models explain the returns of portfolios, over a time period you choose.
All data comes from **Kenneth French's Data Library**.

## Step 1: Setup and data files
Before running, upload the French library CSV files to the Colab **Files** tab (the `/content` folder),
keeping their original names. The notebook will find them automatically.

| Used for | Name on the website | File to upload |
|---|---|---|
| Factors (required) | Fama/French 3 Factors | `F-F_Research_Data_Factors_CSV.zip` |
| Momentum factor | Momentum Factor (Mom) | `F-F_Momentum_Factor_CSV.zip` |
| Test assets | 25 Portfolios Formed on Size and Book-to-Market (5 x 5) | `25_Portfolios_5x5_CSV.zip` |
| Test assets | 25 Portfolios Formed on Size and Momentum (5 x 5) | `25_Portfolios_ME_Prior_12_2_CSV.zip` |
| Test assets | 25 Portfolios Formed on Size and Investment (5 x 5) | `25_Portfolios_ME_INV_5x5_CSV.zip` |
| Test assets | 25 Portfolios Formed on Size and Operating Profitability (5 x 5) | `25_Portfolios_ME_OP_5x5_CSV.zip` |
| Test assets | 25 Portfolios Formed on Size and Short-Term Reversal (5 x 5) | `25_Portfolios_ME_Prior_1_0_CSV.zip` |
| Test assets | 25 Portfolios Formed on Size and Long-Term Reversal (5 x 5) | `25_Portfolios_ME_Prior_60_13_CSV.zip` |
| Test assets | 17 Industry Portfolios | `17_Industry_Portfolios_CSV.zip` |
| Test assets | 48 Industry Portfolios | `48_Industry_Portfolios_CSV.zip` |

Upload the `.zip` files exactly as downloaded, and only the ones you plan to use. Only the 3 Factors file is required.
Where the site also lists an **[ex. Dividends]** or **[Daily]** version of a dataset, use the plain monthly one.

Upload only the ones you plan to use. Only the factors file is required.
Returns in these files are in **percent per month** (1.25 means 1.25%).

In [ ]:
import io, os, re, glob, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import statsmodels.api as sm          # used later for regressions
from scipy import stats               # used later for test statistics
import matplotlib.pyplot as plt       # used later for charts
from IPython.display import display

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 12)
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = "/content"   # where Colab stores uploaded files

### Load the data files
This step finds the French library files you uploaded to the `/content` folder, unzips them, and reads every monthly table inside each one.
The files have a text preamble and several tables, so the code locates each table, converts the month codes into dates, and turns French's missing-value codes (-99.99 and -999) into blanks.
Only the **Fama/French 3 Factors** file is required, and any file you haven't uploaded is skipped.

In [ ]:
import zipfile

# Dataset name on French's website -> file name (lowercase, no extension, no "_CSV")
EXPECTED = {
    "Fama/French 3 Factors":                                              "f-f_research_data_factors",
    "Momentum Factor (Mom)":                                              "f-f_momentum_factor",
    "25 Portfolios Formed on Size and Book-to-Market (5 x 5)":            "25_portfolios_5x5",
    "25 Portfolios Formed on Size and Momentum (5 x 5)":                  "25_portfolios_me_prior_12_2",
    "25 Portfolios Formed on Size and Investment (5 x 5)":                "25_portfolios_me_inv_5x5",
    "25 Portfolios Formed on Size and Operating Profitability (5 x 5)":   "25_portfolios_me_op_5x5",
    "25 Portfolios Formed on Size and Short-Term Reversal (5 x 5)":       "25_portfolios_me_prior_1_0",
    "25 Portfolios Formed on Size and Long-Term Reversal (5 x 5)":        "25_portfolios_me_prior_60_13",
    "17 Industry Portfolios":                                             "17_industry_portfolios",
    "48 Industry Portfolios":                                             "48_industry_portfolios",
}

def normalise(filename):
    """'25_Portfolios_5x5_CSV (1).zip' -> '25_portfolios_5x5' (Colab adds ' (1)' to re-uploads)."""
    name = re.sub(r"\.(zip|csv)$", "", filename, flags=re.I)
    name = re.sub(r"\s*\(\d+\)$", "", name)
    return re.sub(r"_csv$", "", name, flags=re.I).lower()

def read_text(path):
    """Read a French file whether it is a .zip (as downloaded) or an unzipped .csv."""
    if path.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            member = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
            return z.read(member).decode("latin-1")
    with open(path, encoding="latin-1") as f:
        return f.read()

def parse_french_file(path):
    """French's CSVs hold a text preamble followed by one or more monthly tables.
    Returns (vintage_text, {table_title: DataFrame indexed by month})."""
    lines = read_text(path).splitlines()

    # The header says which CRSP database release the file was built from
    text = " ".join(lines[:15])
    m = re.search(r"using the (\d{6}) CRSP database", text)
    vintage = m.group(1) if m else "not stated in file"

    is_month = lambda s: re.fullmatch(r"\d{6}", s.split(",")[0].strip()) is not None
    tables, i = {}, 1
    while i < len(lines):
        if is_month(lines[i]) and not is_month(lines[i - 1]):
            header = lines[i - 1]
            j = i - 2                                   # table title = last non-blank line above the header
            while j >= 0 and not lines[j].strip():
                j -= 1
            title = lines[j].strip() if j >= 0 else "Table"
            if "created by" in title.lower() or not title:
                title = "Monthly"
            k = i
            while k < len(lines) and is_month(lines[k]):  # data rows end at the first non-month row
                k += 1
            cols = ["Month"] + [c.strip() for c in header.split(",")[1:]]
            df = pd.read_csv(io.StringIO("\n".join(lines[i:k])), header=None, names=cols)
            df["Month"] = pd.to_datetime(df["Month"].astype(str), format="%Y%m").dt.to_period("M")
            df = df.set_index("Month").replace([-99.99, -999], np.nan)   # French's missing-value codes
            tables[title] = df
            i = k
        else:
            i += 1
    return vintage, tables

# Find uploaded files (.zip or .csv) and load the ones we recognise
paths = glob.glob(os.path.join(DATA_DIR, "*.zip")) + glob.glob(os.path.join(DATA_DIR, "*.csv"))
available = {normalise(os.path.basename(p)): p for p in paths}
data = {}
for label, stem in EXPECTED.items():
    if stem in available:
        vintage, tables = parse_french_file(available[stem])
        data[label] = {"path": available[stem], "vintage": vintage, "tables": tables}

if "Fama/French 3 Factors" not in data:
    raise FileNotFoundError("F-F_Research_Data_Factors_CSV.zip was not found in /content. Upload it and re-run.")

missing = [k for k in EXPECTED if k not in data]
print(f"Loaded {len(data)} file(s):")
for k in data: print("  +", k)
print(f"\nNot uploaded (fine if you don't need them): {len(missing)}")

Loaded 4 file(s):
  + Fama/French 3 Factors
  + Momentum Factor (Mom)
  + 25 Portfolios Formed on Size and Book-to-Market (5 x 5)
  + 25 Portfolios Formed on Size and Momentum (5 x 5)

Not uploaded (fine if you don't need them): 6


### Which files are loaded, and how current are they?
This table lists each file in use, the **CRSP database release** it was built from (taken from the file's own header), the months it covers, and when it was saved to this Colab session.
French's files don't record a download date, so "Saved to Colab" is when you uploaded the file.
If **Months behind today** is large or the release is old, download fresh copies from French's Data Library and re-upload them.

In [5]:
today = datetime.now()
rows = []
for label, d in data.items():
    first_table = next(iter(d["tables"].values()))        # for portfolio files this is value-weighted monthly
    last = first_table.index.max()
    rows.append({
        "Dataset": label,
        "File": os.path.basename(d["path"]),
        "CRSP release (YYYYMM)": d["vintage"],
        "First month": str(first_table.index.min()),
        "Last month": str(last),
        "Months": len(first_table),
        "Months behind today": (today.year - last.year) * 12 + (today.month - last.month),
        "Saved to Colab": datetime.fromtimestamp(os.path.getmtime(d["path"])).strftime("%Y-%m-%d %H:%M"),
    })
file_report = pd.DataFrame(rows).set_index("Dataset")
display(file_report)

,File,CRSP release (YYYYMM),First month,Last month,Months,Months behind today,Saved to Colab
Dataset,,,,,,,
Fama/French 3 Factors,F-F_Research_Data_Factors_CSV.zip,202609,1926-07,2026-09,1203,2,2026-11-17 03:09
Momentum Factor (Mom),F-F_Momentum_Factor_CSV.zip,202609,1927-01,2026-09,1197,2,2026-11-17 03:09
25 Portfolios Formed on Size and Book-to-Market (5 x 5),25_Portfolios_5x5_CSV.zip,202609,1926-07,2026-09,1203,2,2026-11-17 03:09
25 Portfolios Formed on Size and Momentum (5 x 5),25_Portfolios_ME_Prior_12_2_CSV.zip,202609,1927-01,2026-09,1197,2,2026-11-17 03:09


### A first look at the data
For each file, this lists the tables inside it and shows the first rows of the first table (only the first 6 columns, to keep it readable).
Check that the column names make sense and that the values look like percent per month, for example 1.25 for 1.25%.
Portfolio files contain several tables, such as value-weighted and equal-weighted returns, and we choose between them in a later step.

In [6]:
for label, d in data.items():
    print(f"\n{'=' * 70}\n{label}  ({os.path.basename(d['path'])})")
    for title, df in d["tables"].items():
        print(f"  - {title}: {df.shape[0]} months x {df.shape[1]} columns")
    first_title, first_df = next(iter(d["tables"].items()))
    print(f"\nFirst rows of '{first_title}':")
    display(first_df.iloc[:, :6].head(3))


Fama/French 3 Factors  (F-F_Research_Data_Factors_CSV.zip)
  - The annual TBill return is compounded from the monthly T-bill rates from January to December.: 1203 months x 4 columns

First rows of 'The annual TBill return is compounded from the monthly T-bill rates from January to December.':


,Mkt-RF,SMB,HML,RF
Month,,,,
1926-07,2.8900,-2.4200,-2.7500,0.2200
1926-08,2.6400,-1.4400,4.1300,0.2500
1926-09,0.3800,-1.3600,0.0500,0.2300



Momentum Factor (Mom)  (F-F_Momentum_Factor_CSV.zip)
  - Missing data are indicated by -99.99 or -999.: 1197 months x 1 columns

First rows of 'Missing data are indicated by -99.99 or -999.':


,Mom
Month,
1927-01,0.5700
1927-02,-1.5100
1927-03,3.5200



25 Portfolios Formed on Size and Book-to-Market (5 x 5)  (25_Portfolios_5x5_CSV.zip)
  - Average Value Weighted Returns -- Monthly: 1203 months x 25 columns
  - Average Equal Weighted Returns -- Monthly: 1203 months x 25 columns
  - Number of Firms in Portfolios: 1203 months x 25 columns
  - Average Market Cap: 1203 months x 25 columns
  - and BE(Fiscal Year t-1) is adjusted for net stock issuance to Dec t-1: 1203 months x 25 columns
  - and BE(Fiscal Year t-1) is adjusted for net stock issuance to Jun t: 1203 months x 25 columns
  - Where Mth is a month from June of t to June of t+1: 759 months x 25 columns

First rows of 'Average Value Weighted Returns -- Monthly':


,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1
Month,,,,,,
1926-07,3.7806,-0.4119,-1.9223,-0.4114,1.9583,1.2118
1926-08,-2.2074,-8.7275,2.4557,1.3500,8.5104,2.3620
1926-09,-6.2460,-0.3260,-6.1982,-1.6318,0.8586,-2.6849



25 Portfolios Formed on Size and Momentum (5 x 5)  (25_Portfolios_ME_Prior_12_2_CSV.zip)
  - Average Value Weighted Returns -- Monthly: 1197 months x 25 columns
  - Average Equal Weighted Returns -- Monthly: 1197 months x 25 columns
  - Number of Firms in Portfolios: 1197 months x 25 columns
  - Average Market Cap: 1197 months x 25 columns
  - Equally-Weighted Average of Prior Returns: 1197 months x 25 columns
  - Value-Weighted Average of Prior Returns: 1197 months x 25 columns

First rows of 'Average Value Weighted Returns -- Monthly':


,SMALL LoPRIOR,ME1 PRIOR2,ME1 PRIOR3,ME1 PRIOR4,SMALL HiPRIOR,ME2 PRIOR1
Month,,,,,,
1927-01,-1.7800,3.0500,3.9500,0.9200,-4.8300,-2.8900
1927-02,6.1700,5.9900,6.3500,8.3800,6.6800,5.0500
1927-03,-3.8800,2.4800,-4.4600,-6.8500,-1.0900,-4.1900


### Clean the tables
This step tidies the data so the rest of the notebook is simple:
- **Fixes the table titles.** Each portfolio file keeps only its two return tables, named *Value-weighted returns* and *Equal-weighted returns*. The factor files keep one table, named *Monthly*. The other tables in the files, such as the number of firms, aren't needed for the tests.
- **Renames the corner portfolios** to the standard pattern, for example `SMALL LoBM` becomes `ME1 BM1` and `SMALL HiPRIOR` becomes `ME1 PRIOR5`. Every 25-portfolio set then has the same 25 consistent column names.

It's safe to run this cell more than once.

In [7]:
FACTOR_LABELS = ["Fama/French 3 Factors", "Momentum Factor (Mom)"]

def clean_title(title):
    """Keep only the return tables, with clear names (None = drop the table)."""
    t = title.lower().replace("-", " ")
    if "value weighted returns" in t:
        return "Value-weighted returns"
    if "equal weighted returns" in t:
        return "Equal-weighted returns"
    return None

corner = re.compile(r"^(SMALL|BIG) (Lo|Hi)([A-Za-z]+)$")
def rename_col(col):
    """'SMALL LoBM' -> 'ME1 BM1', 'BIG HiPRIOR' -> 'ME5 PRIOR5'. Other names are left alone."""
    m = corner.match(col)
    if not m:
        return col
    size, level, kind = m.groups()
    return f"ME{1 if size == 'SMALL' else 5} {kind}{1 if level == 'Lo' else 5}"

for label, d in data.items():
    if label in FACTOR_LABELS:
        d["tables"] = {"Monthly": next(iter(d["tables"].values()))}
    else:
        cleaned = {}
        for title, df in d["tables"].items():
            new_title = clean_title(title)
            if new_title:
                cleaned[new_title] = df.rename(columns=rename_col)
        d["tables"] = cleaned

# Check that every 25-portfolio set now has 25 columns named like 'ME3 BM2'
pattern = re.compile(r"^ME[1-5] [A-Z]+[1-5]$")
for label, d in data.items():
    if label.startswith("25 Portfolios"):
        cols = next(iter(d["tables"].values())).columns
        bad = [c for c in cols if not pattern.match(c)]
        status = "OK" if len(cols) == 25 and not bad else f"CHECK these names: {bad}"
        print(f"{label}: {len(cols)} columns, {status}")

print("\nTables kept per file:")
for label, d in data.items():
    print(f"  {label}: {list(d['tables'])}")

25 Portfolios Formed on Size and Book-to-Market (5 x 5): 25 columns, OK
25 Portfolios Formed on Size and Momentum (5 x 5): 25 columns, OK

Tables kept per file:
  Fama/French 3 Factors: ['Monthly']
  Momentum Factor (Mom): ['Monthly']
  25 Portfolios Formed on Size and Book-to-Market (5 x 5): ['Value-weighted returns', 'Equal-weighted returns']
  25 Portfolios Formed on Size and Momentum (5 x 5): ['Value-weighted returns', 'Equal-weighted returns']


---
# Section 2: Your choices and the tests we run

In this section you choose the **time period**, the **factor model**, and the **test assets** (the portfolios the model must explain).
Everything after it, from the regressions to the charts, runs on those choices.

## What are we testing?

A factor model says that a portfolio's average return in excess of the risk-free rate is fully explained by its exposure to a few risk factors:

$$R_{i,t} - RF_t = \alpha_i + \beta_i\,(Mkt\!-\!RF)_t + s_i\,SMB_t + h_i\,HML_t + m_i\,MOM_t + \varepsilon_{i,t}$$

If the model is right, the intercept $\alpha_i$ should be **zero for every portfolio**. A non-zero alpha means the portfolio earned more or less than its factor exposure justifies, so the model is missing something. The models we can compare are:

| Model | Factors |
|---|---|
| CAPM | Mkt-RF |
| Fama-French 3-factor | Mkt-RF, SMB, HML |
| Carhart 4-factor | Mkt-RF, SMB, HML, MOM |
| Custom | any combination of the factors above |

## Test 1: Are all the alphas jointly zero? (GRS F-test)

The **Gibbons-Ross-Shanken (GRS) F-test** asks whether the alphas of all $N$ portfolios are zero at the same time:

$$H_0:\ \alpha_1 = \alpha_2 = \dots = \alpha_N = 0$$

*Why not just look at each alpha's t-statistic?* With 25 portfolios, a few will look "significant" by chance, and the portfolios' errors are correlated with each other. The GRS test combines everything into one answer that accounts for both. A **small p-value (for example below 0.05) means the model is rejected**: its pricing errors are too large to be luck.

## Test 2: Does adding a factor help? (F-test for added factors)

To see whether an extra factor matters, such as MOM on top of the 3-factor model, we test whether its loading is zero:

$$H_0:\ m_i = 0 \ \text{ for the added factor}$$

A **small p-value means the factor adds explanatory power** and belongs in the model. This is the evidence for or against including momentum. The test is most informative on assets that actually have momentum spread, such as the size-momentum portfolios.

## Two ways to test the risk premiums
Both start from the same first step: the time-series regressions give each portfolio's **betas**. Then we estimate the **price of risk ($\gamma$)** of each factor by regressing portfolio returns on those betas:

$$\bar r_i = \gamma_0 + \gamma_1\,\beta_{i,1} + \dots + \gamma_K\,\beta_{i,K} + e_i$$

**Both methods give the same $\gamma$ estimates. They differ only in the standard errors:**
## Method A, one-shot:
A single regression of the portfolios' average excess returns on their betas, with ordinary OLS standard errors. It treats the $N$ portfolios as independent observations (t-distribution with $N-K-1$ degrees of freedom).
## Method B, Fama-MacBeth:
The same regression run every month, then averaged. The standard error is $\text{std}(\gamma_t)/\sqrt{T}$, which reflects that portfolios move together and the real uncertainty over time (t-distribution with $T-1$ degrees of freedom).

**How to read them:** $\gamma_0$ should be near zero, and each $\gamma_k$ should be close to that factor's average return. If a premium is significant under Method A but not Method B, Method A overstated how precisely it was measured.

## Requirements the notebook will check for you
- The GRS test needs **more months than portfolios plus factors** ($T > N + K$). If your date range is too short for your choices, the notebook will tell you.
- Fama-MacBeth needs enough portfolios for the monthly cross-sectional regression to be meaningful, so there must be more portfolios than factors plus one.
- Portfolio returns from French are raw, so we subtract RF to get excess returns before any regression.

In [9]:
WEIGHTING = "Value-weighted returns"        # or "Equal-weighted returns"
PRESETS = {"FF3": ["MKT", "SMB", "HML"], "CARHART": ["MKT", "SMB", "HML", "MOM"]}   # shortcuts only
MIN_MONTHS = 120      # shortest period the analyst may select (10 years)

def factor_code(col):
    """'Mkt-RF' -> 'MKT', 'Mom' -> 'MOM', 'ST_Rev' -> 'ST_REV'."""
    return re.sub(r"[^A-Z0-9_]", "", col.upper().replace("-RF", ""))

# ---------- Build the factor menu from whatever factor files are loaded ----------
factor_menu, rf_source = {}, None            # code -> (dataset, column in that dataset)
for label in FACTOR_LABELS:
    if label not in data:
        continue
    for col in data[label]["tables"]["Monthly"].columns:
        if col.upper() == "RF":              # RF is not a choice: it is used to build excess returns
            rf_source = rf_source or label
            continue
        code, n = factor_code(col), 1
        base = code
        while code in factor_menu:           # same name in another file -> SMB_2, ...
            n += 1
            code = f"{base}_{n}"
        factor_menu[code] = (label, col)

model_menu = {n: f for n, f in PRESETS.items() if all(c in factor_menu for c in f)}
portfolio_menu = [l for l in data if l not in FACTOR_LABELS and WEIGHTING in data[l]["tables"]]
if not portfolio_menu:
    raise FileNotFoundError("No portfolio file uploaded. Upload at least one portfolio set and re-run the loading cells.")

# ---------- 1) Portfolio set ----------
print("TEST PORTFOLIOS (uploaded sets):")
for i, name in enumerate(portfolio_menu, 1):
    print(f"  {i}) {name}")
while True:
    answer = input("Enter the number of the portfolio set: ").strip()
    if answer.isdigit() and 1 <= int(answer) <= len(portfolio_menu):
        portfolio_choice = portfolio_menu[int(answer) - 1]
        break
    print(f"  ! Enter a number from 1 to {len(portfolio_menu)}.")

# ---------- 2) Factors: any single factor or any combination ----------
print("\nFACTORS available from your uploaded files (RF is added automatically):")
for code, (label, col) in factor_menu.items():
    print(f"  {code:<8} from {label}")
if model_menu:
    print("Presets:", ", ".join(f"{n} = {'+'.join(f)}" for n, f in model_menu.items()))
while True:
    answer = input("Enter factor codes separated by commas (or a preset): ").upper().strip()
    if answer in model_menu:
        factor_choice = model_menu[answer]
        break
    codes = [c for c in re.split(r"[,\s]+", answer) if c]
    unknown = [c for c in codes if c not in factor_menu]
    if not codes:
        print("  ! Enter at least one factor.")
    elif unknown:
        print(f"  ! Not available: {unknown}. Choose from {list(factor_menu)} (upload the factor file if one is missing).")
    elif len(set(codes)) < len(codes):
        print("  ! Each factor can be listed only once.")
    else:
        factor_choice = [c for c in factor_menu if c in codes]     # menu order
        break

# ---------- 3) Dates ----------
port_table = data[portfolio_choice]["tables"][WEIGHTING]
used = [port_table, data[rf_source]["tables"]["Monthly"]]
used += [data[factor_menu[c][0]]["tables"]["Monthly"] for c in factor_choice]

earliest = max([pd.Period("1927-01", "M")] + [t.index.min() for t in used])          # nothing before 1927
latest   = min(next(iter(d["tables"].values())).index.max() for d in data.values())  # earliest 'last month' of all loaded files
print(f"\nDATES: allowed range is {earliest} to {latest} (format YYYY-MM)")

def ask_month(prompt):
    while True:
        try:
            return pd.Period(input(prompt).strip(), freq="M")
        except Exception:
            print("  ! Use the format YYYY-MM, for example 1990-01.")

while True:
    start, end = ask_month("Start month: "), ask_month("End month:   ")
    if start < earliest:
        print(f"  ! Start cannot be before {earliest} for your selection.")
    elif end > latest:
        print(f"  ! End cannot be after {latest}, the last month covered by every loaded file.")
    elif start >= end:
        print("  ! Start must be before end.")
    elif (end - start).n + 1 < MIN_MONTHS:
        print(f"  ! Choose at least {MIN_MONTHS} months ({MIN_MONTHS // 12} years). You chose {(end - start).n + 1}.")
    else:
        break

# ---------- Summary and checks ----------
T, N, K = len(pd.period_range(start, end, freq="M")), port_table.shape[1], len(factor_choice)
SETTINGS = {"portfolios": portfolio_choice, "weighting": WEIGHTING, "start": start, "end": end,
            "factors": {c: factor_menu[c] for c in factor_choice}, "rf_source": rf_source}
print(f"\nSELECTED  portfolios: {portfolio_choice} ({N} portfolios, {WEIGHTING.lower()})")
print(f"          factors: {factor_choice}   RF from: {rf_source}")
print(f"          period: {start} to {end} ({T} months)")
if T <= N + K:
    print(f"WARNING: GRS test needs more than {N + K} months (portfolios + factors); you have {T}. It will be skipped.")
elif T < 3 * (N + K):
    print(f"NOTE: {T} months is short for {N} portfolios and {K} factors. The GRS test tends to over-reject in "
          f"small samples, so treat its p-value with caution (more than {3 * (N + K)} months is safer).")

TEST PORTFOLIOS (uploaded sets):
  1) 25 Portfolios Formed on Size and Book-to-Market (5 x 5)
  2) 25 Portfolios Formed on Size and Momentum (5 x 5)
Enter the number of the portfolio set: 2

FACTORS available from your uploaded files (RF is added automatically):
  MKT      from Fama/French 3 Factors
  SMB      from Fama/French 3 Factors
  HML      from Fama/French 3 Factors
  MOM      from Momentum Factor (Mom)
Presets: FF3 = MKT+SMB+HML, CARHART = MKT+SMB+HML+MOM
Enter factor codes separated by commas (or a preset): CARHART

DATES: allowed range is 1927-01 to 2026-09 (format YYYY-MM)
Start month: 1910-01
End month:   2028-01
  ! Start cannot be before 1927-01 for your selection.
Start month: 1930-01
End month:   1928-01
  ! Start must be before end.
Start month: 1930-01
End month:   1937-01
  ! Choose at least 120 months (10 years). You chose 85.
Start month: 1930-01
End month:   1945-01

SELECTED  portfolios: 25 Portfolios Formed on Size and Momentum (5 x 5) (25 portfolios, value-wei

---
# Section 3: How the analysis is conducted

Once your choices are set (dates, factors, test portfolios), the notebook runs the same sequence of steps for any combination. All returns stay in **percent per month**, which doesn't affect any of the test statistics.

## Step 1: Build the data set
- Keep only the months in your chosen period.
- Compute **excess returns** for each of the $N$ portfolios by subtracting the risk-free rate:

  $$r_{i,t} = R_{i,t} - RF_t$$

- The factors Mkt-RF, SMB, HML, MOM and the others are already excess or zero-cost returns, so they are used as they are. They form the $T \times K$ factor matrix $f_t$.

## Step 2: Time-series regressions (one per portfolio)
For each portfolio $i$, ordinary least squares estimates:

$$r_{i,t} = \alpha_i + \boldsymbol{\beta}_i' \, \mathbf{f}_t + \varepsilon_{i,t}$$

This gives each portfolio's alpha, its factor loadings, their t-statistics, the R², and the monthly residuals $\hat\varepsilon_{i,t}$.

## Step 3: The covariance matrices
Two matrices are needed for the joint test:

- **Residual covariance matrix** $\hat\Sigma$ ($N \times N$): how the pricing errors of the portfolios move together.

  $$\hat\Sigma = \frac{1}{T}\sum_{t=1}^{T} \hat\varepsilon_t \hat\varepsilon_t'$$

- **Factor covariance matrix** $\hat\Omega$ ($K \times K$) and the **factor means** $\hat\mu$ ($K \times 1$):

  $$\hat\mu = \frac{1}{T}\sum_{t} \mathbf{f}_t, \qquad \hat\Omega = \frac{1}{T}\sum_{t} (\mathbf{f}_t - \hat\mu)(\mathbf{f}_t - \hat\mu)'$$

Both divide by $T$ (the maximum-likelihood version), which the exact GRS result requires. Your Excel file divides by $T-1$, so its numbers differ very slightly.

## Step 4: The GRS test statistic
The null hypothesis is that all $N$ alphas are zero. The statistic is:

$$F_{GRS} = \frac{T - N - K}{N}\cdot\frac{\hat\alpha'\,\hat\Sigma^{-1}\hat\alpha}{1 + \hat\mu'\,\hat\Omega^{-1}\hat\mu}$$

- The numerator, $\hat\alpha'\hat\Sigma^{-1}\hat\alpha$, measures the size of the pricing errors, scaled by how noisy and correlated they are.
- The denominator, $1 + \hat\mu'\hat\Omega^{-1}\hat\mu$, is the squared maximum Sharpe ratio the factors can achieve, plus one. Factors that earn a lot per unit of risk get more credit, so the same alphas count for less against them.

## Step 5: The probability distribution and the critical value
If the errors are independent over time and normally distributed, then when the model is true:

$$F_{GRS} \sim F(N,\; T - N - K)$$

That is an F distribution with **$N$ numerator degrees of freedom** (the number of alphas tested) and **$T-N-K$ denominator degrees of freedom** (the information left after estimating the parameters). At significance level $\alpha$ (5% by default), the **critical value** is the point that cuts off the top $\alpha$ of this distribution:

$$F_{crit} = F^{-1}_{1-\alpha}\,(N,\; T-N-K)$$

**Decision rule:** if $F_{GRS} > F_{crit}$ (equivalently, if the p-value is below $\alpha$), the alphas are jointly different from zero and the **model is rejected**. Otherwise we cannot reject it.

If the model is *false*, the statistic follows a **non-central** F distribution, with non-centrality parameter

$$\lambda = \frac{T\,\alpha'\Sigma^{-1}\alpha}{1 + \mu'\Omega^{-1}\mu}$$

The larger $\lambda$ is, the more the distribution shifts to the right, and the more likely we are to detect the model's failure. This probability is the **power** of the test.

## How your choices change the distribution

| Choice | What changes | Effect |
|---|---|---|
| **More portfolios ($N$ up)** | Numerator degrees of freedom up, and denominator down | With plenty of months, the critical value falls (roughly 1.5 for 25 portfolios, against 1.8 for 10). But a large $N$ relative to $T$ leaves few denominator degrees of freedom, which fattens the right tail and raises the critical value. |
| **Longer period ($T$ up)** | Denominator degrees of freedom up | The distribution tightens and the critical value falls toward its large-sample limit. The non-centrality $\lambda$ grows with $T$, so the test has more **power**. |
| **Shorter period ($T$ down)** | Denominator degrees of freedom down | The right tail gets heavier, the critical value rises, and the test is both less powerful and less reliable. The test is only defined when $T > N + K$. |
| **More factors ($K$ up)** | Denominator degrees of freedom down by one per factor | The critical value rises slightly. But factors with a high Sharpe ratio also shrink the alphas and enlarge the denominator term, which usually lowers $F_{GRS}$ by more. |

In practice, the notebook prints $F_{GRS}$, the critical value, the degrees of freedom and the p-value for every model you run, so you can compare them directly.


## Step 6: Risk premiums and their standard errors
With $B$ the $N \times (K+1)$ matrix of ones and betas, the estimates are $\hat\gamma = (B'B)^{-1}B'\bar r$.
- **Method A:** $SE_j = \sqrt{s^2\,[(B'B)^{-1}]_{jj}}$, where $s^2$ is the residual variance of this regression ($N-K-1$ degrees of freedom).
- **Method B:** $SE_j = \text{std}(\gamma_{j,t})/\sqrt{T}$, using the monthly cross-sectional estimates ($T-1$ degrees of freedom).

The t-statistic is $\hat\gamma_j / SE_j$, compared with each method's critical value.

## Assumptions to keep in mind
- The GRS F distribution is exact only if the errors are normal and independent over time. Real returns have fatter tails, so treat p-values near the cutoff with caution.
- Betas are estimated over the whole period and assumed constant, as in the original studies.

### The math toolbox
This cell defines the functions used in the analysis. Running it only defines them and does not calculate anything.

| Function | What it does |
|---|---|
| `prepare_data` | Applies your settings: cuts the period, builds excess returns (portfolio return minus RF) and the factor table, and drops any month with missing values |
| `time_series_regression` | Runs one OLS regression per portfolio. Returns alphas, betas, t-statistics, R², residuals, and the covariance matrices $\hat\Sigma$, $\hat\Omega$ and the factor means $\hat\mu$ |
| `grs_test` | Computes the GRS F-statistic, its degrees of freedom, the critical value and the p-value |
| `fama_macbeth` | Runs the monthly cross-sectional regressions and returns the average risk premiums with their t-statistics, next to each factor's average return |
| `run_model` | Runs all of the above for one factor set and returns a summary, so models can be compared easily |

The significance level is set once in `SIG` (5% by default). Results are in percent per month.

In [12]:
SIG = 0.05      # significance level used for critical values

# ---------- 1) Apply the user's settings to the loaded data ----------
def prepare_data(data, settings):
    """Returns excess portfolio returns (T x N) and the chosen factors (T x K), same months."""
    s, e = settings["start"], settings["end"]
    port = data[settings["portfolios"]]["tables"][settings["weighting"]]
    rf = data[settings["rf_source"]]["tables"]["Monthly"]["RF"]
    factors = pd.concat({code: data[label]["tables"]["Monthly"][col]
                         for code, (label, col) in settings["factors"].items()}, axis=1)

    excess = port.sub(rf, axis=0).loc[s:e]                  # portfolio return minus RF
    factors = factors.loc[s:e]
    idx = excess.dropna().index.intersection(factors.dropna().index)   # months with complete data
    dropped = len(pd.period_range(s, e, freq="M")) - len(idx)
    if dropped:
        print(f"Note: {dropped} month(s) dropped because of missing values.")
    return excess.loc[idx], factors.loc[idx]

# ---------- 2) One OLS regression per portfolio ----------
def time_series_regression(excess, factors):
    Y, F = excess.values, factors.values
    T, N = Y.shape
    K = F.shape[1]
    X = np.column_stack([np.ones(T), F])                    # intercept + factors
    B = np.linalg.lstsq(X, Y, rcond=None)[0]                # (K+1) x N coefficients
    resid = Y - X @ B

    dof = T - K - 1
    s2 = (resid ** 2).sum(axis=0) / dof                     # error variance per portfolio
    XtX_inv = np.linalg.inv(X.T @ X)
    se = np.sqrt(np.outer(np.diag(XtX_inv), s2))            # standard errors, (K+1) x N
    t = B / se
    r2 = 1 - (resid ** 2).sum(axis=0) / ((Y - Y.mean(axis=0)) ** 2).sum(axis=0)

    port_names, fac_names = excess.columns, factors.columns
    return {
        "alpha":   pd.Series(B[0], index=port_names),
        "t_alpha": pd.Series(t[0], index=port_names),
        "betas":   pd.DataFrame(B[1:].T, index=port_names, columns=fac_names),
        "t_betas": pd.DataFrame(t[1:].T, index=port_names, columns=fac_names),
        "r2":      pd.Series(r2, index=port_names),
        "resid":   pd.DataFrame(resid, index=excess.index, columns=port_names),
        "Sigma":   resid.T @ resid / T,                     # residual covariance (N x N), divides by T
        "mu":      F.mean(axis=0),                          # factor means (K)
        "Omega":   np.atleast_2d(np.cov(F.T, ddof=0)),      # factor covariance (K x K), divides by T
        "T": T, "N": N, "K": K, "dof": dof,
    }

# ---------- 3) GRS test: are all alphas jointly zero? ----------
def grs_test(ts, sig=SIG):
    T, N, K = ts["T"], ts["N"], ts["K"]
    df1, df2 = N, T - N - K
    if df2 <= 0:
        return {"F": np.nan, "df1": df1, "df2": df2, "F_crit": np.nan, "p_value": np.nan,
                "reject": None, "note": f"Skipped: needs more than {N + K} months, have {T}."}
    a = ts["alpha"].values
    alpha_term = a @ np.linalg.solve(ts["Sigma"], a)                     # alpha' Sigma^-1 alpha
    sharpe_term = 1 + ts["mu"] @ np.linalg.solve(ts["Omega"], ts["mu"])  # 1 + mu' Omega^-1 mu
    F = (df2 / N) * alpha_term / sharpe_term
    F_crit = stats.f.ppf(1 - sig, df1, df2)
    p = stats.f.sf(F, df1, df2)
    return {"F": F, "df1": df1, "df2": df2, "F_crit": F_crit, "p_value": p,
            "reject": F > F_crit, "note": ""}

# ---------- 4) Cross-sectional step: one-shot OLS (Method A) and Fama-MacBeth (Method B) ----------
def cross_sectional(excess, betas, factors, sig=SIG):
    """Both methods give the SAME gammas (risk premiums); only their standard errors differ."""
    T, N = excess.shape
    K = betas.shape[1]
    B = np.column_stack([np.ones(N), betas.values])         # N x (K+1): intercept + betas
    BtB_inv = np.linalg.inv(B.T @ B)
    A = BtB_inv @ B.T
    names = ["Intercept"] + list(betas.columns)

    gamma = A @ excess.mean().values                        # one regression of average returns on betas
    G = pd.DataFrame((A @ excess.values.T).T, index=excess.index, columns=names)   # monthly gammas (T x K+1)
    # (the average of the monthly gammas equals `gamma`)

    # Method A: ordinary OLS standard errors, treats the N portfolios as independent
    dfA, dfB = N - K - 1, T - 1
    if dfA > 0:
        resid = excess.mean().values - B @ gamma
        se_A = np.sqrt((resid @ resid / dfA) * np.diag(BtB_inv))
        crit_A = stats.t.ppf(1 - sig / 2, dfA)
    else:
        se_A, crit_A = np.full(K + 1, np.nan), np.nan

    # Method B: Fama-MacBeth standard errors, from the month-to-month spread of the gammas
    se_B = G.std(ddof=1).values / np.sqrt(T)
    crit_B = stats.t.ppf(1 - sig / 2, dfB)

    t_A, t_B = gamma / se_A, gamma / se_B
    p_A = 2 * stats.t.sf(np.abs(t_A), dfA) if dfA > 0 else np.full(K + 1, np.nan)
    p_B = 2 * stats.t.sf(np.abs(t_B), dfB)

    dec_A = ["n/a" if np.isnan(t) else ("Significant" if abs(t) > crit_A else "Not significant") for t in t_A]
    dec_B = ["Significant" if abs(t) > crit_B else "Not significant" for t in t_B]
    differ, direction = [], []
    for a, b in zip(dec_A, dec_B):
        if a == "n/a":
            differ.append("n/a"); direction.append("-")
        elif a == b:
            differ.append("No"); direction.append("-")
        else:
            differ.append("Yes")
            direction.append("OLS significant, FM not" if a == "Significant" else "FM significant, OLS not")

    table = pd.DataFrame({
        "Gamma": gamma, "Factor avg return": pd.Series(factors.mean()).reindex(names),
        "SE (OLS)": se_A, "t (OLS)": t_A, "p (OLS)": p_A, "Decision (OLS)": dec_A,
        "SE (FM)": se_B, "t (FM)": t_B, "p (FM)": p_B, "Decision (FM)": dec_B,
        "Differ?": differ, "Direction": direction}, index=names)
    return {"table": table, "gammas": G, "crit_A": crit_A, "crit_B": crit_B, "dfA": dfA, "dfB": dfB}

# ---------- 5) Run everything for one factor set ----------
def run_model(excess, factors, sig=SIG):
    ts = time_series_regression(excess, factors)
    grs = grs_test(ts, sig)
    cs = cross_sectional(excess, ts["betas"], factors, sig)
    t_crit = stats.t.ppf(1 - sig / 2, ts["dof"])
    summary = {
        "Factors": "+".join(factors.columns),
        "Months (T)": ts["T"], "Portfolios (N)": ts["N"],
        "GRS F": grs["F"], "F critical": grs["F_crit"], "GRS p-value": grs["p_value"],
        "Avg |alpha| (%/mo)": ts["alpha"].abs().mean(),
        "Avg R2": ts["r2"].mean(),
        "Alphas significant (t-test)": int((ts["t_alpha"].abs() > t_crit).sum()),
    }
    return {"ts": ts, "grs": grs, "cs": cs, "summary": summary}

## Section 4: Run the analysis and keep every result
This cell runs the analysis on the settings you chose and **stores the results in a log called `RESULTS`**, so you can run as many combinations of portfolios and factors as you like and compare them afterwards.

**To add another run:** go back to the settings cell, make new choices, then run this cell again. Each run gets a number (`Run 1`, `Run 2`, ...) and keeps its full output: the settings, the regression table, the GRS test, the Fama-MacBeth premiums, and which data releases were used. Running the same settings twice updates the existing run instead of creating a duplicate.

`RESULTS` lives in this session's memory, so it is lost if the Colab runtime restarts. The comparison table is also saved to a CSV file in the Files tab each time.

### How the two methods are compared
For each run, the table shows the **GRS test** (are all alphas zero?) and the **risk premiums** from the two methods, which have identical estimates and different standard errors.
- Both **t critical values** are shown, since they differ with their degrees of freedom.
- **Methods differ?** counts the coefficients (intercept and factors) whose significance changes between the methods.
- **Direction** says which way: "OLS significant, FM not" means the one-shot method overstated its confidence.

The coefficient-by-coefficient table for the current run appears above the comparison table and is stored with the run.

In [13]:
# The log of all runs. Created once; re-running this cell never erases earlier runs.
if "RESULTS" not in globals():
    RESULTS = {}

def short_name(label):
    return label.replace("Portfolios Formed on ", "").replace(" (5 x 5)", "")

def ts_table(ts):
    """One tidy table per run: alpha, its t-stat, the factor loadings and R2 for each portfolio."""
    return pd.concat([ts["alpha"].rename("alpha"), ts["t_alpha"].rename("t(alpha)"),
                      ts["betas"].add_prefix("beta_"), ts["r2"].rename("R2")], axis=1)

def store_run(settings, model, data):
    """Save one run in RESULTS (replacing it if the same settings were run before). Returns the run id."""
    key = (settings["portfolios"], settings["weighting"], tuple(settings["factors"]),
           str(settings["start"]), str(settings["end"]))
    existing = [rid for rid, r in RESULTS.items() if r["key"] == key]
    run_id = existing[0] if existing else f"Run {len(RESULTS) + 1}"
    files_used = [settings["portfolios"], settings["rf_source"]] + [lab for lab, _ in settings["factors"].values()]
    RESULTS[run_id] = {
        "key": key,
        "portfolios": short_name(settings["portfolios"]),
        "weighting": settings["weighting"].replace(" returns", ""),
        "factors": list(settings["factors"]),
        "start": str(settings["start"]), "end": str(settings["end"]),
        "model": model,                                   # full output: ts, grs, cs, summary
        "ts_table": ts_table(model["ts"]),
        "data_releases": {f: data[f]["vintage"] for f in dict.fromkeys(files_used)},
        "run_time": datetime.now().strftime("%Y-%m-%d %H:%M"),
    }
    return run_id

def comparison_table():
    """One row per stored run: GRS test, both t critical values, and where the two methods disagree."""
    rows = []
    for run_id, r in RESULTS.items():
        s, g, cs = r["model"]["summary"], r["model"]["grs"], r["model"]["cs"]
        tab = cs["table"]
        verdict = "Skipped" if g["reject"] is None else ("Rejected" if g["reject"] else "Not rejected")
        n_diff = int((tab["Differ?"] == "Yes").sum())
        which = "; ".join(f"{name}: {d}" for name, d in tab["Direction"].items() if d != "-") or "-"
        rows.append({"Run": run_id, "Portfolios": r["portfolios"], "Weighting": r["weighting"],
                     "Factors": s["Factors"], "Period": f"{r['start']} to {r['end']}", "Months": s["Months (T)"],
                     "GRS F": s["GRS F"], "F critical": s["F critical"], "GRS p-value": s["GRS p-value"], "GRS verdict": verdict,
                     "t crit (OLS)": cs["crit_A"], "t crit (FM)": cs["crit_B"],
                     "Methods differ?": f"{n_diff} of {len(tab)} coefficients" if n_diff else "No",
                     "Direction": which,
                     "Avg |alpha|": s["Avg |alpha| (%/mo)"], "Avg R2": s["Avg R2"], "Sig. alphas": s["Alphas significant (t-test)"]})
    return pd.DataFrame(rows).set_index("Run")

# ---------- Run the analysis on the current settings ----------
excess, factors = prepare_data(data, SETTINGS)
model = run_model(excess, factors, SIG)
run_id = store_run(SETTINGS, model, data)
r, cs, g = RESULTS[run_id], model["cs"], model["grs"]

# ---------- Show this run ----------
print(f"{run_id}: {r['portfolios']} | factors {'+'.join(r['factors'])} | {r['start']} to {r['end']} | {model['ts']['T']} months")
print(f"Data releases (CRSP): {r['data_releases']}")
if g["reject"] is None:
    print(f"GRS test: {g['note']}")
else:
    print(f"GRS test: F = {g['F']:.3f}, critical value = {g['F_crit']:.3f} (df = {g['df1']}, {g['df2']}), "
          f"p-value = {g['p_value']:.4f} -> model {'REJECTED' if g['reject'] else 'not rejected'} at {SIG:.0%}")

print(f"\nRisk premiums (gamma), percent per month. Same gammas, two sets of standard errors:")
print(f"  OLS one-shot: t critical = {cs['crit_A']:.3f} (df = {cs['dfA']})   |   Fama-MacBeth: t critical = {cs['crit_B']:.3f} (df = {cs['dfB']})")
display(cs["table"].round(4))
print("\nTime-series regressions, first 10 portfolios (full table is stored in the run):")
display(r["ts_table"].head(10).round(3))

# ---------- Compare every run so far ----------
print(f"\nALL RUNS SO FAR ({len(RESULTS)}):")
summary_all = comparison_table()
display(summary_all.round(4))
summary_all.to_csv("/content/results_summary.csv")

Note: 1 month(s) dropped because of missing values.
Run 1: 25 Size and Momentum | factors MKT+SMB+HML+MOM | 1930-01 to 1945-01 | 180 months
Data releases (CRSP): {'25 Portfolios Formed on Size and Momentum (5 x 5)': '202609', 'Fama/French 3 Factors': '202609', 'Momentum Factor (Mom)': '202609'}
GRS test: F = 1.576, critical value = 1.579 (df = 25, 151), p-value = 0.0506 -> model not rejected at 5%

Risk premiums (gamma), percent per month. Same gammas, two sets of standard errors:
  OLS one-shot: t critical = 2.086 (df = 20)   |   Fama-MacBeth: t critical = 1.973 (df = 179)


,Gamma,Factor avg return,SE (OLS),t (OLS),p (OLS),Decision (OLS),SE (FM),t (FM),p (FM),Decision (FM),Differ?,Direction
Intercept,1.1403,NaN,0.7256,1.5715,0.1318,Not significant,1.0748,1.0609,0.2901,Not significant,No,-
MKT,-0.4466,0.5427,0.6362,-0.7019,0.4908,Not significant,1.1758,-0.3798,0.7046,Not significant,No,-
SMB,0.8115,0.8126,0.1582,5.1286,0.0001,Significant,0.4219,1.9233,0.0560,Not significant,Yes,"OLS significant, FM not"
HML,0.9909,0.6344,0.4313,2.2978,0.0325,Significant,0.6880,1.4403,0.1515,Not significant,Yes,"OLS significant, FM not"
MOM,0.1826,0.1157,0.1152,1.5846,0.1287,Not significant,0.6019,0.3034,0.7620,Not significant,No,-



Time-series regressions, first 10 portfolios (full table is stored in the run):


,alpha,t(alpha),beta_MKT,beta_SMB,beta_HML,beta_MOM,R2
ME1 PRIOR1,0.6530,1.6700,0.8020,1.8500,0.6060,-0.5980,0.9360
ME1 PRIOR2,0.5780,1.7050,0.9390,1.6930,0.8090,-0.1890,0.9480
ME1 PRIOR3,0.7370,1.8410,0.8840,1.6020,0.6770,-0.1130,0.9130
ME1 PRIOR4,0.4600,0.9350,1.1440,1.8360,0.4550,0.1590,0.8800
ME1 PRIOR5,0.4570,0.9410,1.0540,1.5080,0.5490,0.4770,0.8410
ME2 PRIOR1,-0.1160,-0.4630,1.0490,1.1300,0.4290,-0.5690,0.9680
ME2 PRIOR2,0.0180,0.0890,1.0250,1.1390,0.3950,-0.2360,0.9710
ME2 PRIOR3,-0.0380,-0.1490,1.0220,0.8640,0.3520,-0.0180,0.9410
ME2 PRIOR4,-0.0590,-0.2250,1.0330,1.1980,0.2410,0.1780,0.9360
ME2 PRIOR5,0.2430,1.0840,1.0270,1.0040,0.5180,0.4940,0.9470



ALL RUNS SO FAR (1):


,Portfolios,Weighting,Factors,Period,Months,GRS F,...,t crit (FM),Methods differ?,Direction,Avg |alpha|,Avg R2,Sig. alphas
Run,,,,,,,,,,,,,
Run 1,25 Size and Momentum,Value-weighted,MKT+SMB+HML+MOM,1930-01 to 1945-01,180,1.5765,...,1.9733,2 of 5 coefficients,"SMB: OLS significant, FM not; HML: OLS signifi...",0.2274,0.9413,1


### Results in plain words
This cell writes a short plain-language summary of a run: what the GRS test concluded, how the Fama-MacBeth risk premiums look, and whether switching from ordinary OLS standard errors to Fama-MacBeth changed any conclusion.

- `explain_run()` summarizes the most recent run.
- `explain_run("Run 2")` summarizes a particular run.
- `explain_all()` summarizes every run you have stored.

In [14]:
import textwrap

def _p(p):
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

def explain_run(run_id=None, sig=SIG):
    if not RESULTS:
        raise ValueError("No runs stored yet. Run the analysis cell first.")
    run_id = run_id or list(RESULTS)[-1]
    r = RESULTS[run_id]
    m, g, cs, ts, s = r["model"], r["model"]["grs"], r["model"]["cs"], r["model"]["ts"], r["model"]["summary"]
    tab = cs["table"]
    T, N, K = ts["T"], ts["N"], ts["K"]
    level = f"{sig:.0%}"
    paras = []

    # --- Setup ---
    paras.append(f"{run_id} tests the {' + '.join(r['factors'])} factor model on the {N} {r['weighting'].lower()} "
                 f"{r['portfolios']} portfolios from {r['start']} to {r['end']} ({T} months).")

    # --- GRS test: are the alphas jointly zero? ---
    if g["reject"] is None:
        paras.append(f"The joint alpha (GRS) test could not be run: {g['note']}")
    else:
        detail = (f"{s['Alphas significant (t-test)']} of the {N} portfolios have individually significant alphas, "
                  f"and the average absolute alpha is {s['Avg |alpha| (%/mo)']:.2f}% per month.")
        stat = f"F = {g['F']:.2f} against a critical value of {g['F_crit']:.2f}, p = {_p(g['p_value'])}"
        if g["reject"]:
            paras.append(f"The GRS test rejects the model at the {level} level ({stat}). In plain words, the pricing errors "
                         f"(alphas) left over after accounting for these factors are too large, taken together, to be "
                         f"explained by chance, so these factors do not fully explain the portfolios' average returns. {detail}")
        else:
            paras.append(f"The GRS test does not reject the model at the {level} level ({stat}). In plain words, the alphas are "
                         f"small enough, taken together, to be consistent with zero, so the data give no statistical reason "
                         f"to say these factors are missing something. This does not prove the model is right, only that "
                         f"this sample cannot show it is wrong. {detail}")

    # --- Fama-MacBeth risk premiums ---
    b = tab.loc["Intercept"]
    sig_word = lambda d: "statistically significant" if d == "Significant" else "not statistically significant"
    fm_text = (f"In the Fama-MacBeth analysis, the average intercept is {b['Gamma']:.2f}% per month "
               f"(t = {b['t (FM)']:.2f}, {sig_word(b['Decision (FM)'])}). If the model were complete this should be zero, "
               + ("so a significant intercept means some average return remains unexplained by the factors. "
                  if b["Decision (FM)"] == "Significant" else "and it is consistent with that. "))
    parts = []
    for name in [n for n in tab.index if n != "Intercept"]:
        row = tab.loc[name]
        parts.append(f"{name} earns a premium of {row['Gamma']:.2f}% per month (the factor's own average return is "
                     f"{row['Factor avg return']:.2f}%; t = {row['t (FM)']:.2f}, {sig_word(row['Decision (FM)'])})")
    paras.append(fm_text + "Each factor's premium is the average reward investors received per unit of exposure to it: "
                 + "; ".join(parts) + ".")

    # --- Does the standard error method change the conclusion? ---
    if (tab["Differ?"] == "n/a").all():
        paras.append("The ordinary OLS comparison could not be computed, because it needs more portfolios than "
                     "factors plus one.")
    else:
        ratio = (tab["SE (FM)"] / tab["SE (OLS)"]).mean()
        diff = tab[tab["Differ?"] == "Yes"]
        if diff.empty:
            paras.append(f"Switching from ordinary OLS standard errors to Fama-MacBeth did not change any conclusion: the "
                         f"same coefficients are significant under both methods. On average the Fama-MacBeth standard "
                         f"errors are {ratio:.1f} times the OLS ones.")
        else:
            bits = []
            for name, row in diff.iterrows():
                k = row["SE (FM)"] / row["SE (OLS)"]
                if row["Direction"].startswith("OLS"):
                    bits.append(f"{name} looks significant with OLS (t = {row['t (OLS)']:.2f}) but not with Fama-MacBeth "
                                f"(t = {row['t (FM)']:.2f}), because its standard error is {k:.1f} times larger")
                else:
                    bits.append(f"{name} is significant with Fama-MacBeth (t = {row['t (FM)']:.2f}) but not with OLS "
                                f"(t = {row['t (OLS)']:.2f})")
            meaning = ("This means the one-shot method overstated how precisely these premiums are known, because it "
                       "treats the portfolios as independent when they actually move together. The Fama-MacBeth result is "
                       "the more reliable one. "
                       if (diff["Direction"].str.startswith("OLS")).any() else
                       "Here the one-shot method understated the evidence, so the Fama-MacBeth result is the more reliable one. ")
            paras.append("Switching from OLS to Fama-MacBeth standard errors changed the conclusion for "
                         f"{len(diff)} of {len(tab)} coefficients: " + "; ".join(bits) + ". " + meaning)

    # --- Reliability note ---
    note = "These are statistical results from one sample, not proof."
    if T < 3 * (N + K):
        note = (f"With only {T} months for {N} portfolios and {K} factors, the tests have limited reliability, especially "
                f"the GRS test. ") + note
    paras.append(note)

    print(f"===== {run_id} =====")
    for p in paras:
        print(textwrap.fill(p, 100))
        print()

def explain_all():
    for rid in RESULTS:
        explain_run(rid)

explain_run()      # summary of the most recent run

===== Run 1 =====
Run 1 tests the MKT + SMB + HML + MOM factor model on the 25 value-weighted 25 Size and Momentum
portfolios from 1930-01 to 1945-01 (180 months).

The GRS test does not reject the model at the 5% level (F = 1.58 against a critical value of 1.58, p
= 0.051). In plain words, the alphas are small enough, taken together, to be consistent with zero,
so the data give no statistical reason to say these factors are missing something. This does not
prove the model is right, only that this sample cannot show it is wrong. 1 of the 25 portfolios have
individually significant alphas, and the average absolute alpha is 0.23% per month.

In the Fama-MacBeth analysis, the average intercept is 1.14% per month (t = 1.06, not statistically
significant). If the model were complete this should be zero, and it is consistent with that. Each
factor's premium is the average reward investors received per unit of exposure to it: MKT earns a
premium of -0.45% per month (the factor's own average r

In [15]:
print("MODEL USED")
print("  Claude Sonnet 5.5 (model ID: claude-sonnet-5-5), by Anthropic\n")

print("USAGE (as of the development session, 55 requests)")
print(f"  Input tokens (fresh):        {188:>12,}")
print(f"  Input tokens (cache write):  {163_871:>12,}")
print(f"  Input tokens (cache read):   {8_764_614:>12,}")
print(f"  Output tokens:               {109_605:>12,}\n")

print("RESOURCES")
resources = [
    ("Data", "Kenneth R. French Data Library: https://mba.tuck.dartmouth.edu/pages/faculty/ken.French/data_library.html"),
    ("Data", "Class Excel file: CAPM_FF3F_Raw_Data_SizeBM.xlsx (1963-02 to 1993-01), used as a check"),
    ("Method", "Fama and French (1993), Common risk factors in the returns on stocks and bonds, Journal of Financial Economics"),
    ("Method", "Carhart (1997), On persistence in mutual fund performance, Journal of Finance"),
    ("Method", "Jegadeesh and Titman (1993), Returns to buying winners and selling losers, Journal of Finance"),
    ("Method", "Gibbons, Ross and Shanken (1989), A test of the efficiency of a given portfolio, Econometrica"),
    ("Method", "Fama and MacBeth (1973), Risk, return, and equilibrium: empirical tests, Journal of Political Economy"),
    ("Software", "Python: pandas, numpy, scipy, statsmodels, matplotlib"),
]
for kind, text in resources:
    print(f"  [{kind}] {text}")

MODEL USED
  Claude Sonnet 5.5 (model ID: claude-sonnet-5-5), by Anthropic

USAGE (as of the development session, 55 requests)
  Input tokens (fresh):                 188
  Input tokens (cache write):       163,871
  Input tokens (cache read):      8,764,614
  Output tokens:                    109,605

RESOURCES
  [Data] Kenneth R. French Data Library: https://mba.tuck.dartmouth.edu/pages/faculty/ken.French/data_library.html
  [Data] Class Excel file: CAPM_FF3F_Raw_Data_SizeBM.xlsx (1963-02 to 1993-01), used as a check
  [Method] Fama and French (1993), Common risk factors in the returns on stocks and bonds, Journal of Financial Economics
  [Method] Carhart (1997), On persistence in mutual fund performance, Journal of Finance
  [Method] Jegadeesh and Titman (1993), Returns to buying winners and selling losers, Journal of Finance
  [Method] Gibbons, Ross and Shanken (1989), A test of the efficiency of a given portfolio, Econometrica
  [Method] Fama and MacBeth (1973), Risk, return, and 